# Train FusionMLP on Extracted Features

**Prerequisite:** Features extracted by Process_All_255 (all batches)

**Input:** `{vid}_features.npy` + `{vid}_labels.npy` per video
**Output:** Trained model + CV F1 + IoU evaluation

In [ ]:
# Cell 1: Setup — Mount Drive + Find Features
import os, json, warnings, shutil
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score

# Mount Drive (features are stored here)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/standup4ai'
FEATURE_DIR = f'{BASE}/features_255'
WORK = '/content/train_output'
os.makedirs(WORK, exist_ok=True)

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Verify features exist
if not os.path.exists(FEATURE_DIR):
    raise FileNotFoundError(f'Features not found at {FEATURE_DIR}. Run Process_All_255_Colab.ipynb first!')

feat_files = sorted([f for f in os.listdir(FEATURE_DIR) if '_features.npy' in f])
print(f'Available features: {len(feat_files)} videos')
assert len(feat_files) >= 10, 'Need at least 10 videos to train'

In [ ]:
# Cell 2: Load All Features
X_list, y_list, vids_list = [], [], []

for feat_file in feat_files:
    vid = feat_file.replace('_features.npy', '')
    label_file = f'{FEATURE_DIR}/{vid}_labels.npy'
    if not os.path.exists(label_file):
        continue
    
    X = np.load(f'{FEATURE_DIR}/{feat_file}')
    y = np.load(label_file)
    
    X_list.append(X)
    y_list.append(y)
    vids_list.extend([vid] * len(y))

X_all = np.vstack(X_list)
y_all = np.concatenate(y_list)
groups = np.array(vids_list)

pos_rate = y_all.mean()
print(f'Total: {len(y_all)} words from {len(set(vids_list))} videos')
print(f'Positive rate: {100*pos_rate:.1f}% ({y_all.sum()} laugh words)')
print(f'Feature dim: {X_all.shape[1]}')

In [ ]:
# Cell 3: Train FusionMLP (5-fold GroupKFold)
class FusionMLP(nn.Module):
    def __init__(self, input_dim=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.3),
            nn.Linear(64, 1))
    def forward(self, x): return self.net(x)

# Auto pos_weight (capped at 3.0)
pos_weight = min((1.0 - pos_rate) / max(pos_rate, 1e-6), 3.0)
print(f'pos_weight: {pos_weight:.2f}')

gkf = GroupKFold(n_splits=5)
models, scalers, fold_f1s = [], [], []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_all, y_all, groups)):
    print(f'\n=== Fold {fold+1}/5 ===')
    Xtr, Xte = X_all[tr_idx], X_all[te_idx]
    ytr, yte = y_all[tr_idx], y_all[te_idx]
    
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr).astype(np.float32)
    Xte_s = scaler.transform(Xte).astype(np.float32)
    
    model = FusionMLP().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    pw = torch.tensor([pos_weight], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    
    Xtr_t = torch.tensor(Xtr_s).to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1).to(device)
    
    best_f1, patience, no_imp = 0, 5, 0
    for epoch in range(50):
        model.train()
        perm = torch.randperm(len(Xtr_t)).to(device)
        for i in range(0, len(Xtr_t), 256):
            idx = perm[i:i+256]
            opt.zero_grad()
            loss = criterion(model(Xtr_t[idx]), ytr_t[idx])
            loss.backward()
            opt.step()
        
        model.eval()
        with torch.no_grad():
            logits = model(torch.tensor(Xte_s).to(device)).squeeze().cpu().numpy()
            probs = 1/(1+np.exp(-logits))
            f = f1_score(yte, (probs >= 0.5).astype(int), zero_division=0)
        if f > best_f1:
            best_f1 = f; no_imp = 0
        else:
            no_imp += 1
        if no_imp >= patience: break
    
    # Final eval
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(Xte_s).to(device)).squeeze().cpu().numpy()
        probs = 1/(1+np.exp(-logits))
        p = precision_score(yte, (probs>=0.5).astype(int), zero_division=0)
        r = recall_score(yte, (probs>=0.5).astype(int), zero_division=0)
        f = f1_score(yte, (probs>=0.5).astype(int), zero_division=0)
    print(f'  F1={f:.4f} P={p:.4f} R={r:.4f}')
    
    models.append(model.cpu())
    scalers.append(scaler)
    fold_f1s.append(f)

print(f'\n{"="*50}')
print(f'CV F1: {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f}')
print(f'Folds: {[round(f,4) for f in fold_f1s]}')

In [ ]:
# Cell 4: Save Model + Results
best_idx = int(np.argmax(fold_f1s))
torch.save(models[best_idx].state_dict(), f'{BASE}/models/fusion255_model.pt')

results = {
    'n_videos': len(set(vids_list)),
    'n_words': int(len(y_all)),
    'positive_rate': float(pos_rate),
    'pos_weight': float(pos_weight),
    'cv_f1': float(np.mean(fold_f1s)),
    'cv_std': float(np.std(fold_f1s)),
    'fold_f1s': [float(f) for f in fold_f1s],
}
os.makedirs(f'{BASE}/models', exist_ok=True)
with open(f'{BASE}/models/fusion255_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('✅ Model and results saved')